# Open source model and data reviews, for variants / AEs

- Beginning with protein structure NER model, sourced from https://huggingface.co/PDBEurope/BiomedNLP-PubMedBERT-ProteinStructure-NER-v3.1

Entities covered by Melanie's model: Bond Interaction, Chemical, Complex Assembly, Evidence, Experimental Method, Gene, Mutant, Oligomeric State, Protein, Protein State, Protein Type, PTM, Residue Name, Residue Name Number, Residue Number, Residue Range, Site, Species, Structure Element, Taxonomy Domain.

In [2]:
from var_utils import *

In [3]:
## Papers sourced via Claude search in ePMC, selected for having a variety of styles of genetic variant mentions
papers = ["PMC7334197", "PMC12713268", "PMC12465344", "PMC12859152", "PMC12874668"]
papers

['PMC7334197', 'PMC12713268', 'PMC12465344', 'PMC12859152', 'PMC12874668']

In [9]:
import sys, os
HERE = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else os.getcwd()
sys.path.insert(0, os.path.normpath(os.path.join(HERE, '../../../epmc-tools/europmc_dev_tool')))
from spacy_patterns import patterns

In [116]:
# Map look-alike characters -> ASCII equivalents
HOMOGLYPH_MAP = str.maketrans({
    0x0441: 'c',   # Cyrillic с → c
    0x0440: 'p',   # Cyrillic р → p
    0x0435: 'e',   # Cyrillic е → e
    0x0430: 'a',   # Cyrillic а → a
    0x2013: '-',   # en-dash → hyphen
    0x2212: '-',   # minus sign → hyphen
    0x2014: '-',   # em-dash → hyphen
})

def map_to_ascii(text: str) -> str:
    """ 
    Replace homoglyph characters with ASCII equivalents.
    Catches chars which may block terms from being caught by regex.
    E.g. 'с.181T >G', wherein 'с' is a Cyrillic character U+0441.
    This can be confused for ASCII 'c', character U+0063
    """
    return text.translate(HOMOGLYPH_MAP)


In [120]:
import json
import re
import uuid

"""
ePMC patterns file defines database cross-references based on regex patterns
Those useful for variant capture include:
- refsnp (dbSNP)
- gca (INSDC genome assembly accessions)
"""
## Compilation of patterns for HGVS statements
# ── Amino acid codes ────────────────────────────────────────────────────────
_AA3 = r'(?:Ala|Arg|Asn|Asp|Cys|Gln|Glu|Gly|His|Ile|Leu|Lys|Met|Phe|Pro|Ser|Thr|Trp|Tyr|Val|Ter|Sec)'
_AA1 = r'[ACDEFGHIKLMNPQRSTVWY]'

# ── Nucleotide change types (shared by g. c. n. r.) ─────────────────────────
# _NUC_POS   = r'[0-9]+(?:[ \t]*[+\-][ \t]*[0-9]+)?'           # e.g. 76, 1799+2
_NUC_POS = r'(?:\*|-)?[0-9]+(?:[ \t]*[+\-][ \t]*[0-9]+)?'
_NUC_RANGE = r'[0-9]+(?:[ \t]*[+\-][ \t]*[0-9]+)?[ \t]*_[ \t]*[0-9]+(?:[ \t]*[+\-][ \t]*[0-9]+)?' # e.g. 76_78
_NUC_BASE  = r'[ACGTUacgtu]'
_SUBST = rf'{_NUC_POS}\s*{_NUC_BASE}\s*>\s*{_NUC_BASE}' # substitution:  A>T, considerate of () // whitespace

_NUC_CHANGE = (
    r'(?:'
    rf'{_SUBST}'
    rf'|(?:{_NUC_RANGE}|{_NUC_POS})[ \t]*delins[ \t]*{_NUC_BASE}+' # indel:    76_78delinsAT
    rf'|(?:{_NUC_RANGE}|{_NUC_POS})[ \t]*del[ \t]*{_NUC_BASE}*'    # deletion: 76del / 76_78del
    rf'|(?:{_NUC_RANGE}|{_NUC_POS})[ \t]*dup[ \t]*{_NUC_BASE}*'    # duplication
    rf'|{_NUC_RANGE}[ \t]*ins[ \t]*{_NUC_BASE}+'                   # insertion: 76_77insA
    rf'|{_NUC_RANGE}[ \t]*inv'                                     # inversion
    r')'
)

# ── Per-prefix patterns ──────────────────────────────────────────────────────
HGVS_GENOMIC  = rf'g\.{_NUC_CHANGE}'                # g.140453136A>T
HGVS_CODING   = rf'c\.{_NUC_CHANGE}'                # c.1799T>A
HGVS_NCRNA    = rf'n\.{_NUC_CHANGE}'                # n.45G>C
HGVS_RNA      = rf'r\.{_NUC_CHANGE}'                # r.76a>u

HGVS_PROTEIN  = (
    r'\(?p\.\(?(?:'                                  # (p.X) and p.(X) handled
    rf'{_AA3}[0-9]+(?:'
        rf'{_AA3}(?:fs(?:Ter[0-9]+)?)?'              # p.Val600Glu / p.Val600GlufsTer5
        rf'|\*'                                      # p.Arg213*  (stop gained)
        rf'|='                                       # p.Val600=  (synonymous)
        rf'|del'                                     # p.Val600del
        rf'|dup'                                     # p.Val600dup
    r')'
    rf'|{_AA1}[0-9]+[{_AA1[1:-1]}*=]'              # p.V600E / p.R213*  (1-letter)
    r')'
)

# ── Optional accession prefix: NM_004333.6: or ENST00000288135.6: ───────────
_ACCESSION = r'(?:\(?[ \t]*(?:NM|NR|NP|NC|NG|XM|XR|ENST|ENSP)_?[0-9]+(?:\.[0-9]+)?[ \t]*\)?[ \t]*:)?'

# ── Combined pattern ─────────────────────────────────────────────────────────
HGVS = re.compile(
    _ACCESSION +
    rf'(?:{HGVS_GENOMIC}|{HGVS_CODING}|{HGVS_NCRNA}|{HGVS_RNA}|{HGVS_PROTEIN})'
)

# Star allele suffix — what follows the gene name
_STAR_SUFFIX = re.compile(
    r'\s*\*\s*[0-9]+[A-Za-z]?(?::[0-9]+[A-Za-z]?)*'          # *13, *02:01
    r'(?:\s*\+\s*\*\s*[0-9]+[A-Za-z]?(?::[0-9]+[A-Za-z]?)*)*' # optional: + *2
)

def find_star_alleles(text: str, gene_spans: list[tuple]) -> list[tuple]:
    """
    For identified geneprotein spans, check if star allele notation
    immediately follows. Returns (text, start, end, label) tuples.
    gene_spans: list of (word, start, end, label)
    """
    results = []
    for word, g_start, g_end, label in gene_spans:
        m = _STAR_SUFFIX.match(text, g_end)   # anchored at gene end position
        if m:
            full_span = text[g_start:m.end()]
            results.append((full_span, g_start, m.end(), "Variant"))
        else:
            results.append((word, g_start, g_end, label))
    return results


In [94]:
from transformers import AutoTokenizer
# Load the same tokenizer the pipeline uses
_tokenizer = AutoTokenizer.from_pretrained("pruas/BENT-PubMedBERT-NER-Gene")

def run_gene_ner_chunked(text: str, pipe, max_tokens: int = 400) -> list[dict]:
    """
    Chunk text at token boundaries (not char boundaries) to stay safely
    under the 512-token position embedding limit.
    """
    # Tokenize with char offset mapping, no special tokens
    enc = _tokenizer(
        text,
        return_offsets_mapping=True,
        add_special_tokens=False
    )
    token_ids   = enc["input_ids"]
    offset_map  = enc["offset_mapping"]   # (char_start, char_end) per token
    results = []
    for chunk_start in range(0, len(token_ids), max_tokens):
        chunk_offsets = offset_map[chunk_start : chunk_start + max_tokens]
        char_start    = chunk_offsets[0][0]
        char_end      = chunk_offsets[-1][1]
        chunk_text    = text[char_start:char_end]
        for ent in pipe(chunk_text):
            ent = dict(ent)
            ent["start"] += char_start   # remap to original text offsets
            ent["end"]   += char_start
            results.append(ent)
    return results


In [121]:
from transformers import pipeline

all_tasks = [] # All LabelStudio tasks

### ------ TODO - Should replace BERT model with ePMC data ---------
# BENT-PubMedBERT, labels: gene, protein
ner_gene = pipeline(
    "ner",
    model="pruas/BENT-PubMedBERT-NER-Gene",
    aggregation_strategy="simple"   # merges subword tokens into full spans
)

label_list = ["refsnp", "gca"] # Relevant patterns
var_patterns = [x for x in patterns if x['label'] in label_list]

for pmcid in papers:
    xml_out = get_epmc_full_text(pmcid=pmcid)
    if xml_out is None:
        continue
    parsed  = parse_epmc_xml(pmcid=pmcid, xml_text=xml_out)


    for sec in parsed.sections:
        ls_tasks = [] # Tasks for paper 'pmcid'
        epmc_matches = []

        text = '\n'.join(sec['text'].split(sep='. ')) # Prepare so as to not mess span numbers
        text = map_to_ascii(text) # Catch non-ASCII chars

        for p in var_patterns:
            pattern = p['pattern']
            epmc_matches.extend([(m.group(), m.start(), m.end(), "Variant") for m in re.finditer(pattern, text)])
        matches = [(m.group(), m.start(), m.end(), "Variant") for m in HGVS.finditer(text)]
        matches.extend(epmc_matches) # Gathering matches

        # Add geneprotein matches
        bert_results = run_gene_ner_chunked(text, ner_gene)
        bert_results = [(ent['word'], ent['start'], ent['end'], "GeneProtein") for ent in bert_results]

        # Catch star allele descriptions if present as 'Variant', otherwise keep as 'GeneProtein'
        star_checked = find_star_alleles(text, bert_results)
        matches.extend(star_checked)

        # Add to LabelStudio-compliant json
        for m in matches:
            ls_tasks.append({
                "value": {
                    "start": m[1],
                    "end":   m[2],
                    "text":  m[0],
                    "labels": [m[3]]
                },
                "id":        str(uuid.uuid4())[:10],
                "from_name": "label",
                "to_name":   "text",
                "type":      "labels",
                "origin":    "prediction"  # marks as pre-annotation, not human
            })

        all_tasks.append({
            "data": {
                "text":    text,
                "heading": sec["heading"] or "Body",
                "pmcid":   pmcid,
            },
            "annotations": [
                {
                    "result":       ls_tasks,
                    "was_cancelled": False,
                    "ground_truth": False,
                }
            ]
        })


        ## Just in case - note for doc-level annotation spans instead of by section
        # offset = 0
        #     for sec in results:
        #         for m in HGVS.finditer(sec.text):
        #             doc_start = offset + m.start()
        #             doc_end   = offset + m.end()
        #         offset += len(sec.text)
        ## Old - printout text and matches
        # hgvs = HGVS.findall(sec["text"])
        # if matches:
        #     sec_text = '\n'.join(sec['text'].split(sep='. '))
        #     print(f"\n[{pmcid}] {sec['heading'] or 'Body'}")
        #     print(f"------SECTION TEXT-----\n{sec_text}\n-----------")
        #     for match, start, end in matches:
        #         print(f"  [{start}:{end}]  {match!r}")
        #         print("- - -")
        # else:
        #     continue

output_path = f"./output/genevar_outputs/variants_ls.json"
with open(output_path, "w") as f:
    json.dump(all_tasks, f, indent=2)
print(f"Written {len(all_tasks)} tasks to {output_path}")

Device set to use mps:0
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Written 81 tasks to ./output/genevar_outputs/variants_ls.json


In [123]:
# # Note written case like 'BRAF V600E mutation' not covered
# Extra format examples
extras = [
    "g.140453136A>T",
    "n.45G>C",
    "r.76a>u",
    "c.76_78del",
    "c.76_77insACG",
    "p.Arg213*",
    "p.Val600=",
    "p.V600E",
    "ENST00000288135.6:c.1799T>A",
    '( NM_001184.4 ):c.4405 A > G',
    'p.(Thr1469Ala)',
    '( NM_139076.3 ):c.727 C > G',
    'p.(Leu243Val)',
    '( NM_004448.4 ):c.3565G > C',
    'p.(Gly1189Arg)',
    '( NM_023110.3 ):c.386 A > C',
    'p.(Asp129Ala)',
    '( NM_007194.4 ):c.906 A > C',
    'p.(Glu302Asp)',
    'c.3306 C > G',
    '*1/*1  +  c.940   C  >  A '
]
print("\n-------MORE EXAMPLES--------")
for e in extras:
    m = HGVS.search(e)
    print(f"  {'OK' if m else 'X'}  {e!r:40s}  -> {m.group() if m else 'no match'}")


-------MORE EXAMPLES--------
  OK  'g.140453136A>T'                          -> g.140453136A>T
  OK  'n.45G>C'                                 -> n.45G>C
  OK  'r.76a>u'                                 -> r.76a>u
  OK  'c.76_78del'                              -> c.76_78del
  OK  'c.76_77insACG'                           -> c.76_77insACG
  OK  'p.Arg213*'                               -> p.Arg213*
  OK  'p.Val600='                               -> p.Val600=
  OK  'p.V600E'                                 -> p.V600E
  OK  'ENST00000288135.6:c.1799T>A'             -> ENST00000288135.6:c.1799T>A
  OK  '( NM_001184.4 ):c.4405 A > G'            -> ( NM_001184.4 ):c.4405 A > G
  OK  'p.(Thr1469Ala)'                          -> p.(Thr1469Ala
  OK  '( NM_139076.3 ):c.727 C > G'             -> ( NM_139076.3 ):c.727 C > G
  OK  'p.(Leu243Val)'                           -> p.(Leu243Val
  OK  '( NM_004448.4 ):c.3565G > C'             -> ( NM_004448.4 ):c.3565G > C
  OK  'p.(Gly1189Arg)'          

## NER model testing

In [59]:
BENT_variant = "alvaroalon2/biobert_genetic_ner"
tokenizer, model, id2label = load_model(model_name = BENT_variant)

all_results: dict[str, list[SectionResult]] = {}
context_words = 10
lookback = None # De-duplicates sentences

for pmcid in papers:
    xml_out = get_epmc_full_text(pmcid=pmcid)
    if xml_out is None:
        continue
    parsed  = parse_epmc_xml(pmcid=pmcid, xml_text=xml_out)
    results = run_ner_pipeline(parsed, model=model, tokenizer=tokenizer, id2label=id2label)
    all_results[pmcid] = results
    # preview annotations
    for sec in results:
        if sec.entities:
            for ent in sec.entities:
                context = get_context(sec.text, ent, window=context_words)
                if context == lookback:
                    continue
                print(f"\n[{pmcid}] {sec.heading or 'Body'}")
                print(f"  {ent.label:15s}  {context}")
                lookback = context

/Users/withers/GitProjects/OTAR3088/ner-model/lib/python3.10/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)



[PMC7334197] Main
  GENETIC          …sequencing projects have revealed a surprisingly high burden of natural ***pLoF*** variation in the human population, including stop-gained, essential splice, and…

[PMC7334197] Main
  GENETIC          …in identifying potential therapeutic targets: confirmed LoF variants in the ***PCSK9 gene*** have been causally linked to low levels of low-density lipoprotein…

[PMC7334197] Main
  GENETIC          …PCSK9 gene have been causally linked to low levels of ***low-density lipoprotein cholesterol 6*** , and have ultimately led to the development of several…

[PMC7334197] Main
  GENETIC          …have ultimately led to the development of several inhibitors of ***PCSK9*** that are now in clinical use for the reduction of…

[PMC7334197] Main
  GENETIC          …the reduction of cardiovascular disease risk. A systematic catalogue of ***pLoF variants*** in humans and the classification of genes along a spectrum…

[PMC7334197] Main
  GENETIC          …rearran

### Testing annotation of ~variants with PDBe model

In [3]:
pdbe_model = "PDBEurope/BiomedNLP-PubMedBERT-ProteinStructure-NER-v3.1"
tokenizer, model, id2label = load_model(model_name = pdbe_model)

all_results: dict[str, list[SectionResult]] = {}
context_words = 10
lookback = None # De-duplicates sentences

for pmcid in papers:
    xml_out = get_epmc_full_text(pmcid=pmcid)
    if xml_out is None:
        continue
    parsed  = parse_epmc_xml(pmcid=pmcid, xml_text=xml_out)
    results = run_ner_pipeline(parsed, model=model, tokenizer=tokenizer, id2label=id2label)
    all_results[pmcid] = results
    # preview annotations
    for sec in results:
        if sec.entities:
            for ent in sec.entities:
                if ent.label == 'mutant':
                    context = get_context(sec.text, ent, window=context_words)
                    if context == lookback:
                        continue
                    print(f"\n[{pmcid}] {sec.heading or 'Body'}")
                    print(f"  {ent.label:15s}  {context}")
                    lookback = context

/Users/withers/GitProjects/OTAR3088/ner-model/lib/python3.10/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)



[PMC12713268] Abstract
  mutant           …had pathogenic variants in the BRCA2 gene. The BRCA1 variant ***c.5266dup*** accounted for 66.2% of all pathogenic variant detected. BRCA1/2 pathogenic…

[PMC12713268] Genetic testing
  mutant           …Collectively, these kits target BRCA1 c.68_69delAG (185delAG), c.181T > G ***(p.Cys61Gly),*** c.1961del (2080delA), c.3700_3704del (3819del5), c.3756_3759del (3875del4), c.4035delA (4153delA), c.5266dupC (5382insC),…

[PMC12713268] Discussion
  mutant           …BRCA1–10 .29- 27%; BRCA2–0 .3-8%) [ 22 ]. The variant ***с.5266dup*** is predominant and is found in 57–68%, the c.4153del pathogenic…

[PMC12713268] Discussion
  mutant           …variant с.5266dup is predominant and is found in 57–68%, the ***c.4153del*** pathogenic variant ranks second - in 12.5% of all BRCA-associated…

[PMC12713268] Discussion
  mutant           …showed that four pathogenic variants were more frequent ( BRCA1 ***с.5266dup,*** с.181T >G, с.1961del, с.4035del) and 

### Testing annotation of ~variants with OTAR3088 model

- Aware variant terms in text are being annotated with different labels, such as 'residue_name_number' at present

In [4]:
otar_model = "Mardiyyah/variant_tapt_freeze_llrd_LR_5e"
tokenizer, model, id2label = load_model(model_name = otar_model)

all_results: dict[str, list[SectionResult]] = {}
context_words = 10
lookback = None # De-duplicates sentences

for pmcid in papers:
    xml_out = get_epmc_full_text(pmcid=pmcid)
    if xml_out is None:
        continue
    parsed  = parse_epmc_xml(pmcid=pmcid, xml_text=xml_out)
    results = run_ner_pipeline(parsed, model=model, tokenizer=tokenizer, id2label=id2label)
    all_results[pmcid] = results
    # Preview annotations
    for sec in results:
        if sec.entities:
            for ent in sec.entities:
                context = get_context(sec.text, ent, window=context_words)
                if context == lookback:
                    continue
                if ent.label in ['mutant', 'residue_name_number']:
                    print(f"\n[{pmcid}] {sec.heading or 'Body'}")
                    print(f"  {ent.label:15s}  {context}")
                lookback = context


[PMC12713268] Genetic testing
  residue_name_number  …Collectively, these kits target BRCA1 c.68_69delAG (185delAG), c.181T > G ***(p.Cys61Gly),*** c.1961del (2080delA), c.3700_3704del (3819del5), c.3756_3759del (3875del4), c.4035delA (4153delA), c.5266dupC (5382insC),…

[PMC12465344] Pathogenic and likely pathogenic variants
  residue_name_number  …Colon Cancer MLH1 Missense ( NM_000249.4 ):c.350 C > T ***p.(Thr117Met)*** Pathogenic rs63750781 47.3% No P2 Male/66 Colon Cancer MLH1 Missense…

[PMC12465344] Pathogenic and likely pathogenic variants
  residue_name_number  …Colon Cancer MLH1 Missense ( NM_000249.4 ):c.244 A > G ***p.(Thr82Ala)*** Pathogenic rs587778998 46.1% No P3 Female/48 Right Colon Cancer PMS2…

[PMC12465344] Pathogenic and likely pathogenic variants
  residue_name_number  …MLH1 Missense ( NM_000249.4 ):c.244 A > G p.(Thr82Ala) Pathogenic ***rs587778998*** 46.1% No P3 Female/48 Right Colon Cancer PMS2 Nonsense (…

[PMC12465344] Pathogenic and likely pathogenic varian

### Testing geneprotein model
- Due to limited coverage of ePMC annotations API, trying out BERT models trained to annotate geneprotein 

In [ ]:
from transformers import pipeline

# BENT-PubMedBERT
# Labels: gene, protein
ner_gene = pipeline(
    "ner",
    model="pruas/BENT-PubMedBERT-NER-Gene",
    aggregation_strategy="simple"   # merges subword tokens into full spans
)

text = """
Pathogenic variants in MLH1 were identified in three patients. 
A frameshift variant in MSH6 was found in P4. Two germline variants 
in APC were detected, and a variant in MUTYH was confirmed in P11 
diagnosed with MUTYH-associated polyposis. Additionally, a splice variant 
in FANCC was detected and a missense variant in CHEK2 was identified.
"""

results = model_pipe(text)
for ent in results:
    print(f"  [{ent['start']:4d}:{ent['end']:4d}]  {ent['entity_group']:12s}  {ent['score']:.3f}  {ent['word']!r}")


Device set to use mps:0


<class 'list'>
  [  24:  28]  B             0.991  'mlh1'
  [  89:  93]  B             0.963  'msh6'
  [ 137: 140]  B             0.963  'apc'
  [ 173: 178]  B             0.872  'mutyh'
  [ 279: 284]  B             0.907  'fancc'
  [ 324: 329]  B             0.982  'chek2'


/Users/withers/GitProjects/OTAR3088/ner-model/lib/python3.10/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


### Comparison of models

In [ ]:
import pandas as pd

def spans_overlap(a: Entity, b: Entity) -> bool:
    """True if two entity spans overlap at all."""
    return a.start < b.end and b.start < a.end

def compare_models(section_results_a: list[SectionResult],
                   section_results_b: list[SectionResult],
                   model_a_name: str = "Model A",
                   model_b_name: str = "Model B",
                   context_words: int = 5,) -> pd.DataFrame:
    """SectionResult: pmcid, heading (section heading), text, entities: list[Entity]"""

    rows = []
    b_index = {(s.pmcid, s.heading): s.entities for s in section_results_b}
    for sect_a in section_results_a:
        key = (sect_a.pmcid, sect_a.heading)
        b_entities = b_index.get(key, [])
        matched_b: set[int] = set()
        for ent_a in sect_a.entities:
            context_a = get_context(sect_a.text, ent_a, window=context_words)
            match_b = None
            for i, ent_b in enumerate(b_entities):
                if i not in matched_b and spans_overlap(ent_a, ent_b):
                    match_b = (i, ent_b)
                    break
            if match_b:
                idx_b, ent_b = match_b
                matched_b.add(idx_b)
                context_b = get_context(sect_a.text, ent_b, window=context_words)
                rows.append({
                    "pmcid":                       sect_a.pmcid,
                    "section":                     sect_a.heading,
                    "span_text":                   ent_a.text,
                    f"{model_a_name}_label":       ent_a.label,
                    f"{model_a_name}_context":     context_a,          # A context
                    f"{model_b_name}_label":       ent_b.label,
                    f"{model_b_name}_context":     context_b,          # B context
                    "label_agreement":             "AGREE" if ent_a.label == ent_b.label else "DISAGREE",
                    "found_by":                    "both",
                })
            else:
                rows.append({
                    "pmcid":                       sect_a.pmcid,
                    "section":                     sect_a.heading,
                    "span_text":                   ent_a.text,
                    f"{model_a_name}_label":       ent_a.label,
                    f"{model_a_name}_context":     context_a,          # A context
                    f"{model_b_name}_label":       "—",
                    f"{model_b_name}_context":     "—",            # B missed it
                    "label_agreement":             "—",
                    "found_by":                    model_a_name,
                })
        for i, ent_b in enumerate(b_entities):
            if i not in matched_b:
                context_b = get_context(sect_a.text, ent_b, window=context_words)
                rows.append({
                    "pmcid":                       sect_a.pmcid,
                    "section":                     sect_a.heading,
                    "span_text":                   ent_b.text,
                    f"{model_a_name}_label":       "—",
                    f"{model_a_name}_context":     "—",            # A missed it
                    f"{model_b_name}_label":       ent_b.label,
                    f"{model_b_name}_context":     context_b,          # B context
                    "label_agreement":             "—",
                    "found_by":                    model_b_name,
                })
    return pd.DataFrame(rows)

In [ ]:
papers = ["PMC7334197", "PMC12713268", "PMC12465344", "PMC12859152", "PMC12874668"]
# Parse: pmcid -> ParsedPaper
all_parsed: dict[str, ParsedPaper] = {}
for pmcid in papers:
    xml_out = get_epmc_full_text(pmcid=pmcid)
    if xml_out is None:
        continue
    all_parsed[pmcid] = parse_epmc_xml(pmcid=pmcid, xml_text=xml_out)

pdbe_model = "PDBEurope/BiomedNLP-PubMedBERT-ProteinStructure-NER-v3.1"
otar_model = "Mardiyyah/variant_tapt_freeze_llrd_LR_5e"
# Run pipelines separately with each model
tokenizer, model, id2label = load_model(pdbe_model)
results_a = {pmcid: run_ner_pipeline(parsed, tokenizer, model, id2label) for pmcid, parsed in all_parsed.items()}
tokenizer, model, id2label = load_model(otar_model)
results_b = {pmcid: run_ner_pipeline(parsed, tokenizer, model, id2label) for pmcid, parsed in all_parsed.items()}

# Flatten results to lists
flat_a = [sec for secs in results_a.values() for sec in secs]
flat_b = [sec for secs in results_b.values() for sec in secs]
df_comp = compare_models(flat_a, flat_b, model_a_name="PubMed-Mutant-BERT", model_b_name="OTAR-Variant-BERT", context_words=10)

# ── Summary views ──────────────────────────────────────────────────────────────
# Full comparison table
display(df_comp)
df_comp.to_csv("./output/genevar_outputs/df_comp.tsv", sep="\t")
# Agreement rate among spans found by both
both = df_comp[df_comp["found_by"] == "both"]
agree_rate = (both["label_agreement"] == "AGREE").mean()
print(f"\nLabel agreement (overlapping spans): {agree_rate:.1%}")
# What each model found exclusively
print(f"\nOnly by PubMed-Mutant-BERT:  {(df_comp['found_by']=='PubMed-Mutant-BERT').sum()} entities")
print(f"Only by OTAR-Variant-BERT:     {(df_comp['found_by']=='OTAR-Variant-BERT').sum()} entities")
print(f"Found by both:           {len(both)} entities")
# Breakdown by label
display(
    df_comp.groupby(["found_by", "PubMed-Mutant-BERT_label", "OTAR-Variant-BERT_label"])
      .size()
      .reset_index(name="count")
      .sort_values("count", ascending=False)
)

/Users/withers/GitProjects/OTAR3088/ner-model/lib/python3.10/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


,pmcid,section,span_text,PubMed-Mutant-BERT_label,PubMed-Mutant-BERT_context,OTAR-Variant-BERT_label,OTAR-Variant-BERT_context,label_agreement,found_by
0,PMC7334197,Abstract,human,species,"…describe the aggregation of 125,748 exomes an...",species,"…describe the aggregation of 125,748 exomes an...",AGREE,both
1,PMC7334197,Abstract,human,species,…by sequencing and annotation errors. Using an...,species,…by sequencing and annotation errors. Using an...,AGREE,both
2,PMC7334197,Abstract,human,species,…Using an improved model of human mutation rat...,species,…Using an improved model of human mutation rat...,AGREE,both
3,PMC7334197,Abstract,human,species,…validate this classification using data from ...,species,…validate this classification using data from ...,AGREE,both
4,PMC7334197,Abstract,whole-exome,experimental_method,…rare diseases. A catalogue of predicted loss-...,—,—,—,PubMed-Mutant-BERT
...,...,...,...,...,...,...,...,...,...
2819,PMC12874668,Discussion,Xp,—,—,gene,…translocation of the Yp11.2 region (containin...,—,OTAR-Variant-BERT
2820,PMC12874668,Discussion,SOC,—,—,experimental_method,…pathway applicable to “unexplained adverse pr...,—,OTAR-Variant-BERT
2821,PMC12874668,Discussion,SR,—,—,experimental_method,…risk assessment and recommend preimplantation...,—,OTAR-Variant-BERT
2822,PMC12874668,Discussion,short arm,—,—,structure_element,…in sex chromosome analysis—when the Y chromos...,—,OTAR-Variant-BERT



Label agreement (overlapping spans): 81.6%

Only by PubMed-Mutant-BERT:  509 entities
Only by OTAR-Variant-BERT:     883 entities
Found by both:           1432 entities


,found_by,PubMed-Mutant-BERT_label,OTAR-Variant-BERT_label,count
55,both,protein,protein,419
40,both,experimental_method,experimental_method,347
12,OTAR-Variant-BERT,—,residue_number,198
4,OTAR-Variant-BERT,—,gene,158
23,PubMed-Mutant-BERT,protein,—,137
...,...,...,...,...
49,both,mutant,protein_state,1
61,both,protein_state,residue_number,1
28,PubMed-Mutant-BERT,residue_number,—,1
51,both,mutant,residue_name,1


In [30]:
labels_of_interest = {"mutant", "residue_name_number"}
mask = (
    df_comp["PubMedBERT-NER_label"].isin(labels_of_interest) | df_comp["VariantBERT_label"].isin(labels_of_interest))
df_filtered = df_comp[mask]
df_filtered.to_csv("./output/genevar_outputs/varmodels_comparison.tsv", sep="\t")
display(df_filtered)

,pmcid,section,span_text,PubMedBERT-NER_label,PubMedBERT-NER_context,VariantBERT_label,VariantBERT_context,label_agreement,found_by
409,PMC12713268,Abstract,c.5266dup,mutant,…BRCA2 gene. The BRCA1 variant ***c.5266dup***...,residue_number,…BRCA2 gene. The BRCA1 variant ***c.5266dup***...,✗,both
475,PMC12713268,Genetic testing,Cys,mutant,"…c.68_69delAG (185delAG), c.181T > G ***(p.Cys...",residue_name_number,"…c.68_69delAG (185delAG), c.181T > G ***(p.Cys...",✗,both
476,PMC12713268,Genetic testing,61,mutant,"…c.68_69delAG (185delAG), c.181T > G ***(p.Cys...",ptm,"…c.68_69delAG (185delAG), c.181T > G ***(p.Cys...",✗,both
574,PMC12713268,Discussion,с.5266dup,mutant,…[ 22 ]. The variant ***с.5266dup*** is predom...,residue_number,…[ 22 ]. The variant ***с.5266dup*** is predom...,✗,both
575,PMC12713268,Discussion,c.4153del,mutant,"…is found in 57–68%, the ***c.4153del*** patho...",residue_number,"…is found in 57–68%, the ***c.4153del*** patho...",✗,both
...,...,...,...,...,...,...,...,...,...
2315,PMC12859152,CYP2C19 deletion frequency in the Estonian Bi...,CYP2C19*37,mutant,…CYP2C19 partial gene deletion ( ***CYP2C19*37...,protein,…CYP2C19 partial gene deletion ( ***CYP2C19*37...,✗,both
2316,PMC12859152,CYP2C19 deletion frequency in the Estonian Bi...,CYP2C19*42,mutant,…gene deletion ( CYP2C19*37 or ***CYP2C19*42**...,mutant,…gene deletion ( CYP2C19*37 or ***CYP2C19*42**...,✓,both
2396,PMC12859152,Post-recruitment long-read sequencing of parti...,c.-806C>T,mutant,…well-known upstream regulatory variant CYP2C1...,residue_name,…well-known upstream regulatory variant CYP2C1...,✗,both
2401,PMC12859152,Post-recruitment long-read sequencing of parti...,rs,gene,…upstream regulatory variant CYP2C19*17 (c.-80...,residue_name_number,…upstream regulatory variant CYP2C19*17 (c.-80...,✗,both


### Older early works around model testing

In [23]:
def filter_res(results: List[Tuple], filter: str):
    for res in results:
        text = res[0]
        tagged = res[1]
        filtered = [x["entity"] for x in tagged if x["type"] == filter]
        if len(filtered) > 0:
            print("- - - - - - Original text - - - - -")
            pprint([x for x in text.split(sep='. ')])
            print(f"- - Terms tagged of type '{filter}' - -")
            pprint(filtered)

filter_res(e, "mutant")

NameError: name 'e' is not defined

In [ ]:
for p in papers:
    e = run_model(input_text=p)

/Users/withers/GitProjects/OTAR3088/ner-model/lib/python3.10/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [ ]:
# Some gemini example sentences, these are more obvious examples of variants
protein_variant_sentences = [
    "Studies on the protein-protein interaction network revealed that the R175H variant of p53 dramatically alters its binding partners, shifting its function from a tumor suppressor to an oncogenic driver.",
    "The structural consequences of the E6V substitution in hemoglobin are profound, leading to the formation of rigid polymers that deform red blood cells into their characteristic sickle shape.",
    "The D90A mutation in superoxide dismutase 1 (SOD1) highlights the complexity of disease mechanisms, as its pathogenicity in familial ALS is not due to a loss of catalytic activity but rather to a gain of toxic function.",
    "The ΔF508 deletion in the CFTR protein prevents its proper folding, triggering the cell's quality control system to target the misfolded protein for destruction in the endoplasmic reticulum.",
    "Kinase inhibitors are now a standard therapy for melanoma patients with the V600E mutation in BRAF, which constitutively activates the downstream signaling cascade.",
    "The single amino acid change A67T in transthyretin increases its propensity to misfold and aggregate, forming amyloid fibrils that deposit in the heart and other tissues.",
    "The pathogenicity of the A223P variant in fibrillin-1 lies in its disruption of calcium-binding sites, which are critical for the formation of stable microfibrils in the extracellular matrix.",
    "Patients with familial chylomicronemia resulting from the S267L variant of lipoprotein lipase exhibit severely impaired lipid clearance due to the protein's reduced catalytic efficiency.",
    "The R117H amino acid substitution in Factor V makes it resistant to cleavage by activated protein C, a critical step in down-regulating blood coagulation.",
    "The P101S amino acid change in the prion protein (PrP) is a crucial determinant of disease pathology, as it favors the conversion of the normal PrP ($PrP^C$) into the disease-associated scrapie form ($PrP^{Sc}$).",
    "The E280A mutation in presenilin 1 alters the cleavage of the amyloid precursor protein (APP), leading to an overproduction of the highly aggregation-prone Aβ42 peptide.",
    "Achondroplasia is caused by a G380R variant in fibroblast growth factor receptor 3 (FGFR3), which leads to the constitutive activation of the receptor even in the absence of its ligand.",
    "The K456N variant of cardiac troponin I impairs the calcium-dependent regulation of myocardial contraction, which is the underlying cause of hypertrophic cardiomyopathy.",
    "The C278F substitution in fibroblast growth factor receptor 2 (FGFR2) is a well-known example of a gain-of-function mutation, leading to premature fusion of the cranial sutures.",
    "The C282Y variant in the HFE protein disrupts its interaction with the transferrin receptor, leading to a breakdown in the regulation of iron absorption.",
    "The p.L320P variant of lamin A/C causes dilated cardiomyopathy by compromising the structural integrity of the nuclear lamina and disrupting normal gene expression patterns.",
    "The S295C mutation in myotilin creates a new cysteine residue, which can form inappropriate disulfide bonds, leading to the formation of aggregates that interfere with muscle function.",
    "The R402Q variant in the ATP-binding cassette transporter A4 (ABCA4) disrupts its ability to transport retinoids, leading to the accumulation of toxic compounds in the retinal pigment epithelium.",
    "A study of familial ALS patients identified the H46R mutation in SOD1, which was shown to lead to a misfolding and aggregation of the protein that is toxic to motor neurons.",
    "The E258K mutation in lysosomal acid lipase is a clear example of a loss-of-function variant, as it severely impairs the protein's ability to hydrolyze cholesterol esters.",
    "Individuals homozygous for the M34T variant in connexin 26 (GJB2) often present with non-syndromic hearing loss due to the disruption of gap junction function in the inner ear.",
    "Congenital contractural arachnodactyly is associated with the K197R mutation in fibrillin-2, a protein crucial for elastic fiber formation in connective tissues.",
    "The L268P mutation in type II collagen (COL2A1) results in a protein with impaired triple helix formation, leading to the skeletal abnormalities characteristic of Stickler syndrome.",
    "The P1213L variant in dynein heavy chain 1 (DYNC1H1) compromises the protein's motor function, resulting in defects in retrograde axonal transport that cause spinal muscular atrophy.",
    "The R782W mutation in α₁-antitrypsin (SERPINA1) results in a misfolded protein that is trapped in the liver, leading to a lack of circulating α₁-antitrypsin in the lungs.",
    "The R136Q amino acid substitution in ubiquitin carboxyl-terminal hydrolase L1 (UCHL1) affects its deubiquitinating activity, which may contribute to the accumulation of protein aggregates observed in Parkinson's disease.",
    "The P270S variant in glucocerebrosidase (GBA1) has been shown to reduce the enzyme's activity, which is a known risk factor for Parkinson's disease.",
    "The A30P mutation in α-synuclein is a rare but well-documented cause of familial Parkinson's disease, as it promotes the formation of neurotoxic fibrils and Lewy bodies.",
    "The R46L variant in proprotein convertase subtilisin/kexin type 9 (PCSK9) is a functional polymorphism that reduces the protein's ability to degrade LDL receptors, leading to lower levels of circulating LDL cholesterol.",
    "The A152T variant in the protein tau (MAPT) is a known risk factor for frontotemporal dementia, as it enhances the aggregation and misfolding of the protein."
]

In [ ]:
# Gemini mimicking descriptions of paper results, for protein variants then non-coding genetic variants to see what happens
protein_variant_sentences_experimental_results = [
    "Circular dichroism spectroscopy of the R175H p53 variant revealed a significant decrease in α-helical content compared to the wild-type protein, indicating a loss of structural integrity.",
    "In our gel electrophoresis assays, the E6V hemoglobin variant showed a distinct band shift in non-denaturing conditions, consistent with polymer formation under deoxygenated states.",
    "Immunoblot analysis of cells expressing the D90A SOD1 variant demonstrated a marked increase in detergent-insoluble protein, suggesting a propensity for aggregation.",
    "Western blot analysis of the ΔF508 CFTR variant showed a complete absence of the mature, fully glycosylated band, confirming that the protein is not trafficked to the cell surface.",
    "In vitro kinase assays confirmed a 500-fold increase in the phosphorylation of ERK and MEK substrates when using the V600E BRAF variant, demonstrating constitutive activation.",
    "Fluorescence microscopy with Thioflavin S staining revealed abundant amyloid deposits in cardiac tissue sections from mice expressing the A67T transthyretin variant.",
    "Our surface plasmon resonance (SPR) data demonstrated that the A223P fibrillin-1 variant has a significantly reduced binding affinity for calcium ions (Kd = 450 nM vs 50 nM for WT).",
    "Lipolysis assays using the S267L lipoprotein lipase variant showed only 15% of the catalytic activity observed with the wild-type enzyme.",
    "Coagulation assays of plasma containing the R117H Factor V variant showed a 12-fold increase in clotting time after the addition of activated protein C, confirming its resistance to inactivation.",
    "Real-time quaking-induced conversion (RT-QuIC) assays demonstrated that the P101S prion protein variant seeded the rapid formation of fibrils at a concentration 100-fold lower than the wild-type protein.",
    "Cell-free cleavage assays confirmed that the E280A presenilin 1 variant preferentially generated the highly amyloidogenic Aβ42 peptide over the Aβ40 peptide.",
    "Dual-luciferase reporter assays confirmed a 20-fold increase in STAT3 signaling in cells expressing the G380R FGFR3 variant, even in the absence of exogenous FGF ligand.",
    "Calcium-sensitizing assays on skinned cardiac muscle fibers revealed that the K456N troponin I variant increased the calcium sensitivity of force generation by 2.5-fold.",
    "Immunofluorescence staining revealed that the C278F FGFR2 variant localized to the cell membrane at a higher density and for a longer duration than the wild-type protein.",
    "In our iron uptake experiments, cells expressing the C282Y HFE variant failed to show the characteristic down-regulation of transferrin receptor expression in response to high iron levels.",
    "Immunoblotting of nuclear fractions showed a fragmented lamin A/C network in cells from patients with the p.L320P variant, indicating a loss of nuclear integrity.",
    "Pull-down assays with a cysteine-reactive probe demonstrated that the S295C myotilin mutation forms intermolecular disulfide bonds, leading to the formation of high-molecular-weight aggregates.",
    "Liposome-based transport assays using the R402Q ABCA4 variant showed a 90% reduction in the translocation of all-trans-retinal across the membrane.",
    "Immunohistochemistry of spinal cord sections from transgenic mice expressing the H46R SOD1 variant revealed extensive intracellular inclusions in motor neurons, which were immunoreactive for ubiquitin.",
    "In vitro enzyme kinetics demonstrated that the E258K lysosomal acid lipase variant had a Vmax that was less than 5% of the wild-type enzyme, confirming a severe loss-of-function.",
    "Dye-transfer assays showed a complete failure of Lucifer yellow transfer between cells expressing the M34T connexin 26 variant, indicating a loss of gap junction function.",
    "Atomic force microscopy of extracellular matrix fibers from cells expressing the K197R fibrillin-2 variant revealed an aberrant, disorganized microfibril network.",
    "A triple helix formation assay using purified L268P collagen type II demonstrated a significant reduction in the protein's ability to form stable triple helices.",
    "Live-cell imaging of retrograde axonal transport in neurons with the P1213L dynein heavy chain 1 variant showed a 75% decrease in the speed of cargo movement toward the cell body.",
    "Immunofluorescence of liver tissue from a patient with the R782W α₁-antitrypsin variant revealed abundant inclusions of the protein within hepatocytes, confirming its failure to be secreted.",
    "The R136Q UCHL1 variant showed a 5-fold decrease in its isopeptidase activity against a ubiquitinated substrate in a mass spectrometry-based enzymatic assay.",
    "In a fluorometric enzyme assay, the P270S glucocerebrosidase variant exhibited only 40% of the catalytic activity observed with the wild-type protein.",
    "Transmission electron microscopy of neuronal cultures expressing the A30P α-synuclein variant showed the presence of abundant fibrillar aggregates, similar to those found in Lewy bodies.",
    "In our binding assays, the R46L PCSK9 variant demonstrated a 3-fold decrease in its binding affinity for the LDL receptor, leading to a reduced rate of receptor degradation.",
    "Confocal microscopy of cells expressing the A152T tau variant revealed a significant increase in the formation of puncta and filamentous structures, suggesting enhanced aggregation."
]

non_coding_variant_sentences = [
    "Using a dual-luciferase reporter assay, we demonstrated that the SNP rs6983267, located in a long-range enhancer for the MYC oncogene, significantly increased reporter gene expression by 2.5-fold in colon cancer cell lines.",
    "Chromatin immunoprecipitation followed by sequencing (ChIP-seq) for the transcription factor TCF7L2 revealed a loss of binding at the promoter of the TCF7L2 gene in the presence of the rs7903146 risk allele.",
    "Our massively parallel reporter assay (MPRA) data showed that the A>T variant in the 3' UTR of the NOS1AP gene reduced the expression of a linked fluorescent reporter by 40% in cardiomyocyte cells.",
    "Sanger sequencing of the TERT promoter in melanoma samples identified two recurrent non-coding variants, c.-124C>T and c.-146C>T, which were shown by RT-qPCR to increase TERT mRNA expression by 15- and 20-fold, respectively.",
    "A quantitative chromatin conformation capture (3C) assay demonstrated that the rs9349379 SNP in a genomic insulator element disrupts the physical loop connecting the ESR1 promoter to its upstream enhancer.",
    "Minigene splicing assays confirmed that the deep intronic variant IVS21-12C>T in the CFTR gene leads to the creation of a cryptic splice site, resulting in a mis-spliced transcript lacking exon 22.",
    "In a yeast-based screen, we found that a 15-bp deletion within the HBB promoter region completely abolished the binding of the GATA-1 transcription factor, consistent with the observed phenotype in beta-thalassemia patients.",
    "We performed allelic expression analysis on post-mortem brain tissue and found that the rs12913832 SNP in the OCA2 gene enhancer was strongly associated with a 50% decrease in its expression.",
    "Our electro mobility shift assay (EMSA) results showed that the C>G SNP in the MDM2 promoter creates a novel high-affinity binding site for the SP1 transcription factor, leading to increased MDM2 transcription.",
    "RNA-seq analysis of patient-derived lymphoblastoid cell lines with the rs2201397 SNP in the TRIM59 3' UTR showed a significant upregulation of TRIM59 transcript levels, which we hypothesize is due to altered microRNA binding.",
    "Using CRISPR-Cas9 to introduce the rs7552554 variant in the promoter region of the MYB oncogene, we observed a 3-fold increase in H3K27ac histone marks, indicating enhanced enhancer activity.",
    "The 5-kb deletion upstream of the SOX2 gene was found to completely abrogate the expression of the gene in embryonic stem cells, suggesting the removal of a critical long-range enhancer.",
    "In a high-throughput screen, we identified an A>G variant in the promoter of a lncRNA that significantly decreased its transcription by 75% in a disease-relevant cell model.",
    "Our Northern blot analysis revealed that the rs10825036 SNP in the 3' UTR of the MIR-499 gene resulted in a mature miRNA transcript that was 50% less abundant compared to the wild-type allele.",
    "Allele-specific chromatin accessibility assays (ATAC-seq) revealed a loss of an open chromatin region at the locus containing the rs12740374 variant, suggesting a disruption of a transcription factor binding site.",
    "The 3' UTR of the KRAS gene containing the rs61764370 variant was cloned into a reporter plasmid, and subsequent luciferase assays showed a significant reduction in luciferase activity, consistent with altered miRNA-mediated repression.",
    "Chr-RNA FISH experiments confirmed that the C>T mutation in the SHH limb enhancer region failed to form a chromatin loop with the SHH promoter, explaining the observed limb malformations.",
    "Reporter gene assays showed that the rs11603334 SNP in the ARAP1 promoter led to a 2-fold upregulation of gene expression in pancreatic beta cells, which was corroborated by decreased binding of PAX4 and PAX6 transcription factors in EMSA.",
    "We used CRISPR-based tiling scans to systematically test variants in the LDLR promoter, finding that a specific A>C substitution reduced promoter activity by 60%, a key finding for familial hypercholesterolemia.",
    "RNA-seq of patient tumor samples with the rs73183594 variant in the IRF4 enhancer showed a significant increase in IRF4 mRNA expression levels, suggesting a gain-of-function mechanism.",
    "Our allele-specific sequencing analysis of RNA from heterozygous individuals demonstrated that the rs3796548 SNP, a non-coding variant, leads to a 3-fold increase in the expression of the TULP3 gene.",
    "The deletion of an enhancer 150 kb upstream of the GATA6 gene was found to cause a loss of GATA6 protein expression in pancreatic progenitor cells, as confirmed by immunostaining.",
    "Using a high-resolution 4C-seq, we mapped the chromosomal interactions of the genomic region containing a non-coding variant, revealing the creation of a novel chromatin loop between a distant enhancer and an oncogene.",
    "A minigene assay showed that a single nucleotide variant (G>A) in the 5' untranslated region of an unknown gene caused an increase in ribosome readthrough and protein translation.",
    "Quantitative real-time PCR (qRT-PCR) from patient fibroblasts with a deep intronic variant in the IDUA gene revealed a reduced level of the fully spliced transcript and an increase in a mis-spliced isoform.",
    "The C>G variant in the promoter of the CASP8 gene, located in a CpG island, was shown by methylation-specific PCR to be associated with a complete loss of DNA methylation and subsequent gene silencing.",
    "In our chromatin accessibility assay, the A>G SNP in the FTO enhancer was found to disrupt the binding of the ARID5B transcription factor, leading to a downstream effect on gene regulation.",
    "Our reporter assay demonstrated that the rs1403565 variant in the PITX2 enhancer increased the transcriptional activity of a reporter construct by 30% in cardiac tissue cells.",
    "The insertion of a SINE retrotransposon into the promoter of the FGFR2 gene was shown by reporter assays to completely abolish the gene's promoter activity in fibroblasts.",
    "CRISPR activation experiments using a sgRNA targeting the promoter region of a lncRNA confirmed that a non-coding variant within this region was responsible for a significant reduction in gene expression."
]

In [ ]:
gemini_egs = run_model(input_text=protein_variant_sentences)
filter_res(gemini_egs, "mutant")

- - - - - - Original text - - - - -
['Studies on the protein-protein interaction network revealed that the R175H '
 'variant of p53 dramatically alters its binding partners, shifting its '
 'function from a tumor suppressor to an oncogenic driver.']
- - Terms tagged of type 'mutant' - -
['r1', '75', 'h']
- - - - - - Original text - - - - -
['The structural consequences of the E6V substitution in hemoglobin are '
 'profound, leading to the formation of rigid polymers that deform red blood '
 'cells into their characteristic sickle shape.']
- - Terms tagged of type 'mutant' - -
['e6', 'v']
- - - - - - Original text - - - - -
['The D90A mutation in superoxide dismutase 1 (SOD1) highlights the complexity '
 'of disease mechanisms, as its pathogenicity in familial ALS is not due to a '
 'loss of catalytic activity but rather to a gain of toxic function.']
- - Terms tagged of type 'mutant' - -
['d', '90', 'a']
- - - - - - Original text - - - - -
['The ΔF508 deletion in the CFTR protein preve

In [ ]:
gemini_exps = run_model(input_text=protein_variant_sentences_experimental_results)
filter_res(gemini_exps, "mutant")

- - - - - - Original text - - - - -
['Circular dichroism spectroscopy of the R175H p53 variant revealed a '
 'significant decrease in α-helical content compared to the wild-type protein, '
 'indicating a loss of structural integrity.']
- - Terms tagged of type 'mutant' - -
['r1', '75', 'h']
- - - - - - Original text - - - - -
['In our gel electrophoresis assays, the E6V hemoglobin variant showed a '
 'distinct band shift in non-denaturing conditions, consistent with polymer '
 'formation under deoxygenated states.']
- - Terms tagged of type 'mutant' - -
['e6', 'v']
- - - - - - Original text - - - - -
['Immunoblot analysis of cells expressing the D90A SOD1 variant demonstrated a '
 'marked increase in detergent-insoluble protein, suggesting a propensity for '
 'aggregation.']
- - Terms tagged of type 'mutant' - -
['d', '90', 'a']
- - - - - - Original text - - - - -
['Western blot analysis of the ΔF508 CFTR variant showed a complete absence of '
 'the mature, fully glycosylated band, con

In [ ]:
gemini_gene = run_model(input_text=non_coding_variant_sentences)
filter_res(gemini_gene, "mutant")

- - - - - - Original text - - - - -
['Our massively parallel reporter assay (MPRA) data showed that the A>T '
 "variant in the 3' UTR of the NOS1AP gene reduced the expression of a linked "
 'fluorescent reporter by 40% in cardiomyocyte cells.']
- - Terms tagged of type 'mutant' - -
['a > t']
- - - - - - Original text - - - - -
['Sanger sequencing of the TERT promoter in melanoma samples identified two '
 'recurrent non-coding variants, c.-124C>T and c.-146C>T, which were shown by '
 'RT-qPCR to increase TERT mRNA expression by 15- and 20-fold, respectively.']
- - Terms tagged of type 'mutant' - -
['c . - 124 c > t', 'c . - 146 c > t']
- - - - - - Original text - - - - -
['Minigene splicing assays confirmed that the deep intronic variant '
 'IVS21-12C>T in the CFTR gene leads to the creation of a cryptic splice site, '
 'resulting in a mis-spliced transcript lacking exon 22.']
- - Terms tagged of type 'mutant' - -
['iv', 's2', '1 - 12 c > t']
- - - - - - Original text - - - - -
['Our e

In [ ]:
filter_res(gemini_gene, "protein")

- - - - - - Original text - - - - -
['Chromatin immunoprecipitation followed by sequencing (ChIP-seq) for the '
 'transcription factor TCF7L2 revealed a loss of binding at the promoter of '
 'the TCF7L2 gene in the presence of the rs7903146 risk allele.']
- - Terms tagged of type 'protein' - -
['tcf', '7', 'l2', 'tcf', '7', 'l2']
- - - - - - Original text - - - - -
['Our massively parallel reporter assay (MPRA) data showed that the A>T '
 "variant in the 3' UTR of the NOS1AP gene reduced the expression of a linked "
 'fluorescent reporter by 40% in cardiomyocyte cells.']
- - Terms tagged of type 'protein' - -
['nos', '1a', 'p']
- - - - - - Original text - - - - -
['Sanger sequencing of the TERT promoter in melanoma samples identified two '
 'recurrent non-coding variants, c.-124C>T and c.-146C>T, which were shown by '
 'RT-qPCR to increase TERT mRNA expression by 15- and 20-fold, respectively.']
- - Terms tagged of type 'protein' - -
['tert', 'tert']
- - - - - - Original text - - - - -

In [ ]:
gemini_gene

[('Using a dual-luciferase reporter assay, we demonstrated that the SNP rs6983267, located in a long-range enhancer for the MYC oncogene, significantly increased reporter gene expression by 2.5-fold in colon cancer cell lines.',
  [{'entity': 'dual - luciferase reporter assay',
    'type': 'experimental_method'},
   {'entity': 'rs', 'type': 'gene'},
   {'entity': '69', 'type': 'gene'},
   {'entity': '83', 'type': 'gene'},
   {'entity': '267', 'type': 'gene'},
   {'entity': 'long - range enhancer', 'type': 'structure_element'},
   {'entity': 'myc', 'type': 'protein_type'}]),
 ('Chromatin immunoprecipitation followed by sequencing (ChIP-seq) for the transcription factor TCF7L2 revealed a loss of binding at the promoter of the TCF7L2 gene in the presence of the rs7903146 risk allele.',
  [{'entity': 'chromatin immunoprecipitation followed by sequencing',
    'type': 'experimental_method'},
   {'entity': 'chip - seq', 'type': 'experimental_method'},
   {'entity': 'transcription factor', 't

### AE classifier - not worth chasing

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# This model is a classifier: AE is / is not present in text
tokenizer = AutoTokenizer.from_pretrained("jocforero/longformer-adverse-event-classifier")
ae_model = AutoModelForSequenceClassification.from_pretrained("jocforero/longformer-adverse-event-classifier")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/961 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/595M [00:00<?, ?B/s]

In [17]:
gemini_ae_egs = [
    "Gastrointestinal bleeding occurred in two patients receiving the active treatment."
    "One participant in the control group experienced anaphylactic shock requiring immediate medical intervention.",
    "The most frequently reported adverse event was mild-to-moderate headache, observed in 15% of the study population.",
    "Three cases of acute renal failure were documented, leading to discontinuation of the study drug.",
    "A dose-dependent increase in serum transaminases was observed in a subset of participants.",
    "The device's implantation was associated with local tissue necrosis in 5% of subjects.",
    "A single instance of pulmonary embolism was reported in a patient with no prior history of thrombotic events.",
    "Patients on the higher dose of the compound exhibited a significant increase in systolic blood pressure.",
    "Study withdrawal was attributed to unmanageable nausea and vomiting in four subjects.",
    "One participant developed a maculopapular rash that was determined to be a hypersensitivity reaction to the study medication.",
    "This is a test, nothing to see here.", # Should be false, completely out of context
    "The study was conducted in a double-blind, placebo-controlled manner.", # Should be false, in context but 0 AE
    "An adverse event of headache was reported." # Can't get much more obvious a statement
    ]

In [16]:
import torch

batch_size=4
max_length=512

all_results = []
    
# Process in batches
for i in range(0, len(gemini_ae_egs), batch_size):
    # Get the current batch
    batch_texts = gemini_ae_egs[i:i + batch_size]
    
    # Tokenize and pass to model
    inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=max_length)
    print(batch_texts)
    with torch.no_grad():
        outputs = ae_model(**inputs)

    # Decode predictions and configure id to label map
    predictions = torch.argmax(outputs.logits, dim=1)
    print(predictions)
    id_to_label = ae_model.config.id2label
    print(id_to_label)

['Gastrointestinal bleeding occurred in two patients receiving the active treatment.One participant in the control group experienced anaphylactic shock requiring immediate medical intervention.', 'The most frequently reported adverse event was mild-to-moderate headache, observed in 15% of the study population.', 'Three cases of acute renal failure were documented, leading to discontinuation of the study drug.', 'A dose-dependent increase in serum transaminases was observed in a subset of participants.']
tensor([0, 0, 0, 0])
{0: 'LABEL_0', 1: 'LABEL_1'}
["The device's implantation was associated with local tissue necrosis in 5% of subjects.", 'A single instance of pulmonary embolism was reported in a patient with no prior history of thrombotic events.', 'Patients on the higher dose of the compound exhibited a significant increase in systolic blood pressure.', 'Study withdrawal was attributed to unmanageable nausea and vomiting in four subjects.']
tensor([0, 0, 0, 0])
{0: 'LABEL_0', 1: '